# 2次元イジング模型：計算手法の数式解説

このノートでは、シミュレーションコードで実装した物理量とアルゴリズムの数学的背景を解説します。

## 1. モデルの定義

2次元正方格子（サイズ $L \times L$）上の**強磁性イジング模型**を考えます。各サイト $i$ にはスピン変数 $S_i \in \{+1, -1\}$ が割り当てられています。

### ハミルトニアン (エネルギー)
系の全エネルギー $H$ は、隣接するスピン間の相互作用のみを考慮して次のように書けます：
$$ H = -J \sum_{\langle i,j \rangle} S_i S_j $$
ここで、
- $J > 0$: 強磁性相互作用定数（スピンが揃うとエネルギーが下がる）。
- $\sum_{\langle i,j \rangle}$: 隣接するペア（最近接ハミルトニアン）についての和。

## 2. メトロポリス法 (Metropolis Algorithm)

熱平衡状態（温度 $T$）における物理量を求めるため、マルコフ連鎖モンテカルロ法を用います。

### スピン反転に伴うエネルギー変化
あるサイト $i$ のスピン $S_i$ を反転させたとき（$S_i \to -S_i$）、エネルギーの変化 $\Delta E$ はその周囲のスピンのみに依存します：
$$ \Delta E = E_{\text{new}} - E_{\text{old}} = 2J S_i \sum_{j \in \text{neighbors}} S_j $$

### 遷移確率
詳細釣合い条件を満たすように、状態遷移確率 $W$ を以下のように設定します：
$$ W(S_i \to -S_i) = \begin{cases} \exp(-\Delta E / k_B T) & (\Delta E > 0) \\ 1 & (\Delta E \leq 0) \end{cases} $$
※ 実装では計算を簡単にするため $k_B = 1$ としています。

## 3. 1 MCS (Monte Carlo Step) の定義

1ステップの間に、格子の全サイト数 $N = L^2$ 回の更新試行を行います。
- 全サイトを順番に選ぶ、またはランダムに $N$ 回選ぶ方法があります。
- 計算量は $L \times L$ の全サイトを舐めるため、**$O(L^2)$** となります。

## 4. 有限サイズスケーリングと転移温度

無限系では $T_c \approx 2.269 J$ で相転移が起きますが、有限系 $L$ では転移がなだらかになります。これを解析するために**ビンダー累積量 (Binder Cumulant)** を使います。

### ビンダー累積量の定義
磁化 $m = \frac{1}{N}\sum S_i$ に対して、以下のように定義されます：
$$ U_L = 1 - \frac{\langle m^4 \rangle}{3 \langle m^2 \rangle^2} $$

### なぜこれを使うのか？
1. **高温相 ($T > T_c$)**: 磁化分布はガウス分布に近づき、$U_L \to 0$ ($L \to \infty$) となります。
2. **低温相 ($T < T_c$)**: スピンが揃い、磁化分布は $m = \pm 1$ にピークを持つ2峰性になり、$U_L \to 2/3$ となります。
3. **臨界点 ($T = T_c$)**: $U_L$ の値は（主要項において）システムサイズ $L$ に依存しない一定の値をとります。

したがって、**異なる $L$ について $U_L$ をプロットしたとき、すべての曲線が交わる点が転移温度 $T_c$** となります。